In [ ]:
import sys, random, argparse
from pathlib import Path
import numpy as np
sys.path.append(r'E:\Program Files\Coreform Cubit 2025.12\bin')
import cubit
#cubit.init(['cubit','-nojournal'])
cubit.init(['cubit'])
# --- 可见输出工具（即使 cubit.init() 改写了 stdout） ---
def echo(msg):
    sys.__stdout__.write(str(msg) + "\n")
    sys.__stdout__.flush()

: 

In [ ]:
def make_Kalthoff_Winkler_board(x, y, z, hole_gap, hole_width, hole_depth):
    if x <= 0 or y <= 0 or z <= 0:
        raise ValueError("x, y, z must be > 0")
    if hole_gap < 0 or hole_width <= 0 or hole_depth <= 0:
        raise ValueError("hole_gap must be >= 0; hole_width and hole_depth must be > 0")
    if hole_depth > y:
        raise ValueError("hole_depth cannot exceed height y")

    cubit.cmd(f"brick x {x} y {y} z {z}")
    board_id = cubit.get_last_id("volume")

    radius = hole_width * 0.5
    slot_rect_depth = hole_depth - radius
    if slot_rect_depth <= 0:
        raise ValueError("hole_depth must be greater than hole_width/2 for a rounded slot")

    x0 = -0.5 * hole_gap
    x1 = 0.5 * hole_gap
    y_top = 0.5 * y
    z_center = 0.0

    def _make_slot_volume(x_center):
        cubit.cmd(f"brick x {hole_width} y {hole_depth} z {z+2}")
        rect_id = cubit.get_last_id("volume")
        cubit.cmd(
            f"move volume {rect_id} x {x_center} y {y_top - 0.5 * hole_depth} z {z_center}"
        )

        cubit.cmd(f"create cylinder radius {radius} height {z+2}")
        cyl_id = cubit.get_last_id("volume")
        cubit.cmd(
            f"move volume {cyl_id} x {x_center} y {y_top - hole_depth} z {z_center}"
        )

        cubit.cmd(f"unite volume {rect_id} {cyl_id}")
        return cubit.get_last_id("volume")

    slot1_id = _make_slot_volume(x0)
    slot2_id = _make_slot_volume(x1)

    cubit.cmd(f"subtract volume {slot1_id} {slot2_id} from volume {board_id}")
    cubit.cmd(f'delete volume {slot1_id}{slot2_id}')
    print("slot1 exists:", slot1_id in cubit.get_entities("volume"))
    print("slot2 exists:", slot2_id in cubit.get_entities("volume"))

    return board_id



In [ ]:
def make_Kalthoff_Winkler_cylinder(pos_x, pos_y, pos_z, radius, height):
    if radius <= 0 or height <= 0:
        raise ValueError("radius, height must be > 0")
    if pos_y < y/2 + height/2:
        raise ValueError("pos_y too low, risk of overlapping")
    
    cubit.cmd(f"create cylinder radius {radius} height {height}")
    vid = cubit.get_last_id("volume")

    # z 轴 -> y 轴
    cubit.cmd(f"rotate volume {vid} angle -90 about x")
    cubit.cmd(
        f"move volume {vid} x {pos_x} y {pos_y} z {pos_z}"
    )

In [ ]:
# KW板参数示例
x = 200.0
y = 100.0
z = 9.0
hole_gap = 51.5
hole_width = 1.5
hole_depth = 50.0
# KW圆柱参数示例
pos_x = 0
pos_y = 100
pos_z = 0
radius = 25
height = 60

board_id = make_Kalthoff_Winkler_cylinder(pos_x, pos_y, pos_z, radius, height)



In [ ]:


board_id = make_Kalthoff_Winkler_board(x, y, z, hole_gap, hole_width, hole_depth)


In [ ]:
h_global = 5.0

# 1) 先把所有边设为均匀分布（避免残留 bias）
cubit.cmd("curve all scheme equal")

# 2) 全局边长尺度 = 5mm
cubit.cmd(f"curve all size {h_global}")
# 局部加密
ids = [48, 51, 52, 53, 59, 65, 68, 69]
cubit.cmd("curve " + " ".join(map(str, ids)) + " size 0.19")
graded_curves = [47, 49, 54, 56, 58, 60, 66, 70]
h_fine   = 1
h_coarse = 5.0

# 先确保这些曲线启用 bias scheme（指数型分布）
cubit.cmd("curve " + " ".join(map(str, graded_curves)) + " scheme bias")

def vtx_coords(vid: int):
    """
    不同 Cubit 版本接口可能略有差异：
    - 常见：cubit.vertex(vid).coordinates()
    如果你的版本不是这个函数名，把这里替换成你环境可用的取坐标函数即可。
    """
    return cubit.vertex(vid).coordinates()  # -> (x,y,z)

for cid in graded_curves:
    # 取曲线两端点
    vids = cubit.get_relatives("curve", cid, "vertex")
    if len(vids) != 2:
        raise RuntimeError(f"curve {cid} endpoints != 2, got {vids}")

    v1, v2 = vids
    _, y1, _ = vtx_coords(v1)
    _, y2, _ = vtx_coords(v2)

    v_low, v_high = (v1, v2) if y1 <= y2 else (v2, v1)

    # 低 y 端细，高 y 端粗（沿 +y 方向指数加粗）
    cubit.cmd(f"vertex {v_low}  size {h_fine}")
    cubit.cmd(f"vertex {v_high} size {h_coarse}")


In [ ]:
# --------------------------
# Tet meshing (after sizing)
# --------------------------

# 0) 如果之前已经生成过网格，先删掉（避免“看起来没变化”）
cubit.cmd("delete mesh volume all")
cubit.cmd("delete mesh surface all")   # 可选，但建议加上更干净

# 1) 设默认单元类型为 tet（可选，但建议）
cubit.cmd("set default element type tet")

# 2) 体网格方案：tetmesh
cubit.cmd("volume all scheme tetmesh")

# 3) （可选但很重要）给体内部一个尺寸控制，否则只靠边界曲线有时内部会过粗/过细
#    如果你想全局 5mm 作为基准：
cubit.cmd("volume all size 5")

# 4) 生成体网格（会自动先做表面三角网格，再铺四面体）
cubit.cmd("mesh volume all")


In [ ]:

# 导出 .cubit 文件（相对当前工作目录）
cubit.cmd("export cubit 'kw_board.cub5' overwrite")

In [ ]:
cubit.cmd("export mesh 'kw_board.g' overwrite")